In [ ]:
import subprocess
import time
import re
import os

# 1. Install dependencies
!sudo apt-get install -y zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in background
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print("Waiting for Ollama server to start...")
time.sleep(8)

# 3. Verify Ollama is running
import urllib.request
for attempt in range(5):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("✅ Ollama server is UP")
        break
    except Exception:
        print(f"Attempt {attempt+1}: Ollama not ready yet, waiting...")
        time.sleep(5)

# 4. Pull the model
print("Pulling llama2:7b...")
!ollama pull llama2:7b

# 5. Verify model loaded
print("\nTesting model...")
!curl -s http://localhost:11434/api/generate \
  -H "Content-Type: application/json" \
  -d '{"model": "llama2:7b", "prompt": "Hello", "stream": false, "options": {"num_predict": 10}}'

# 6. Download and start Cloudflare tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("\nStarting tunnel...")
tunnel_process = subprocess.Popen(
    [
        "./cloudflared-linux-amd64", "tunnel", 
        "--url", "http://localhost:11434", 
        "--http-host-header", "localhost:11434"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url_found = False
for i in range(30):
    line = tunnel_process.stdout.readline()
    match = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if match:
        print("\n" + "="*50)
        print("✅ SUCCESS! Your API URL is:")
        print(match.group(1))
        print("="*50 + "\n")
        url_found = True
        break

if not url_found:
    print("❌ Could not find tunnel URL.")

!sleep infinity

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
Pulling meditron:7b (This will take 1-2 minutes)...

Starting tunnel...

✅ SUCCESS! Your API URL is:
https://lifestyle-prove-advertisements-paragraph.trycloudflare.com

